In [ ]:
"""
exp01 Step 04: Multivariate Signal Check (DIAGNOSTIC, PROTOCOL.md §13.3; protocol 1.1, decided by Nicolas 2026-10-06)

exp01 is STOPPED (step 02 verdict, p = 0.10). This step was added AFTER exp01's results were known: it can explain the
failure; it CANNOT reverse the STOP or restart exp01's walk-forward.

Question: (a) does the COMBINATION of the 33 model features, through exp01's own LightGBM classifiers (one per distance,
fit on training only), predict "take profit first" better than the base rate on the pooled validation rows, beyond what
chance and the model's flexibility produce? (b) If so, does the top decile of predicted P(TP) clear its break-even rate?

    - Windows, sampling, features and distances exactly as step 02 (asserted against its trial-log entry).
    - Real check: AUC, top-decile TP rate, lift and break-even per distance, block-bootstrap intervals of ISO weeks.
    - Null: the feature columns shifted by whole sessions (step 02's shift), the models RETRAINED, 19 runs.
    - Verdict: INFORMATION only if the real mean AUC over the 21 distances is above EVERY null run (p <= 1/20; ties: none).
      Outcome A (no information), B (information, not tradable), C (information, possibly tradable: candidate pending
      review, the untouched-window decision is parked, no trading experiment is built on it).
    - Nothing on or after 2026-05-14 is loaded (raw bars cut at the last validation date; every row asserted).
One trial-log entry (stage "multivariate_check"). Prerequisite: exp01 step 01. Outputs:
store04_experiments/exp01_minute_entry/step04_multivariate_check_data/.
"""
None

In [ ]:
# IMPORT GENERAL PACKAGES
import json
import time
import pandas as pd
import numpy as np
# IMPORT THE SHARED CONFIGURATION, THE EXPERIMENT CONFIGURATION AND THE PATHS
from so import config, paths
from experiments.exp01_minute_entry import config as exp_config
# IMPORT RAW DATA, TRADE EXECUTION AND FILE FUNCTIONS
from so.core.raw_data import get_complete_ohlcv_pdf
from so.core.trade_execution import get_ohlcv_array_dict
from so.core.local_file_management import get_date_range_file_path_list, write_csv_file_to_path
# IMPORT THE EXPERIMENT FUNCTIONS
from experiments.exp01_minute_entry.model_dataset import get_model_feature_col_str_list
from experiments.exp01_minute_entry.multivariate_check import get_schedule_tuple, get_check_window_dict, read_check_window_pdf_tuple, check_last_date_bool, \
                                                               get_check_lgbm_param_dict, run_model_check_dict, get_null_run_pdf_tuple, \
                                                               get_multivariate_verdict_dict, get_calibration_pdf, get_feature_importance_pdf
# IMPORT TRIAL LOG FUNCTIONS
from so.core.trial_log import get_config_hash_str, log_trial_dict, read_trial_log_pdf, get_trial_count_pdf, print_previous_trial_warning

In [ ]:
# DISPLAY THE EXPERIMENT, THE PROTOCOL VERSION AND THE CONFIGURATION HASH (SHARED + EXPERIMENT CONFIG)
print(f"Experiment:\t{exp_config.EXPERIMENT_NAME}\tProtocol:\t{exp_config.PROTOCOL_VERSION}\tConfig Hash:\t{get_config_hash_str(exp_config)}")
print("exp01 status: STOPPED (step 02, p = 0.10). Step 04 is a diagnostic added after exp01's results: it cannot reverse the STOP.")
# DEFINE THE OUTPUT FOLDER OF THIS STEP
output_path_str = paths.get_experiment_data_path_str(exp_config.EXPERIMENT_NAME, "step04_multivariate_check_data")
print(f"Output folder:\t{output_path_str}")
# CALL FUNCTION TO WARN ABOUT EARLIER ENTRIES OF THIS STAGE (A RERUN ADDS TRIALS)
previous_trial_count = print_previous_trial_warning(exp_config, exp_config.MULTIVARIATE_CHECK_STAGE_STR)

In [ ]:
# COLLECT THE SESSION DATES FROM THE MODEL DATASET FILE NAMES (THE SCHEDULE IS ANCHORED ON THE LAST SESSION, AS STEP 02; NO FILE IS READ HERE)
dataset_file_path_list = get_date_range_file_path_list(paths.get_experiment_data_path_str(exp_config.EXPERIMENT_NAME, "step01_model_dataset_data"))
session_date_list = [pd.to_datetime(f.split("_")[-1].replace(".csv", "")).date() for f in dataset_file_path_list]
# CALL FUNCTION TO GET THE FOLDS AND THE POOLED VALIDATION FOLDS
fold_pdf, pooled_fold_pdf = get_schedule_tuple(session_date_list)
window_dict = get_check_window_dict(pooled_fold_pdf)
# DISPLAY INFORMATION
print(f"Train:\t\t{window_dict['train_start']} -> {window_dict['train_end']}\t({window_dict['train_years']} years)")
for _, row in pooled_fold_pdf.iterrows():
    print(f"Validation:\t{row['valid_start']} -> {row['valid_end']}\t(fold {row['fold_id']})")
print(f"Latest test window (not read):\t{fold_pdf['test_start'].iloc[-1]} -> {fold_pdf['test_end'].iloc[-1]}")
# ASSERT THE WINDOWS ARE THOSE OF STEP 02 (ITS CALIBRATED TRIAL-LOG ENTRY, HASH 15716b01a3)
trial_log_pdf = read_trial_log_pdf()
step02_pdf = trial_log_pdf[(trial_log_pdf["experiment"] == exp_config.EXPERIMENT_NAME) & (trial_log_pdf["stage"] == "signal_check") &
                           (trial_log_pdf["config_hash"] == "15716b01a3")] if not trial_log_pdf.empty else pd.DataFrame()
if len(step02_pdf):
    step02_setting_dict = json.loads(step02_pdf.iloc[-1]["setting_json"])
    assert str(step02_setting_dict["train_start"]) == str(window_dict["train_start"]) and str(step02_setting_dict["train_end"]) == str(window_dict["train_end"])
    assert step02_setting_dict["valid_fold_id_list"] == window_dict["valid_fold_id_list"]
    print(f"✅ Windows identical to step 02 (trial {step02_pdf.iloc[-1]['trial_id']})")
else:
    print("⚠️ No step 02 entry with hash 15716b01a3 in this trial log (smoke run only): windows not compared")
# ASSERT THE LAST VALIDATION DATE IS BEFORE THE UNTOUCHED WINDOW
last_valid_date = window_dict["valid_end"]
assert pd.Timestamp(last_valid_date) < pd.Timestamp(exp_config.MULTIVARIATE_CHECK_UNTOUCHED_START_DATE_STR)
print(f"Last date loaded:\t{last_valid_date}\t(untouched window starts {exp_config.MULTIVARIATE_CHECK_UNTOUCHED_START_DATE_STR})")

In [ ]:
# CALL FUNCTION TO READ THE TRAINING AND POOLED VALIDATION ROWS (AS STEP 02; SORTED BY DECISION TIME)
train_pdf, valid_pdf = read_check_window_pdf_tuple(window_dict, pooled_fold_pdf, exp_config.SIGNAL_CHECK_DELTA_LIST)
# CALL FUNCTION TO READ THE RAW BARS UP TO THE LAST VALIDATION DATE (UNIQUENESS WEIGHTS OF THE TRAINING ROWS)
complete_ohlcv_pdf = get_complete_ohlcv_pdf(cutoff_date_str_in=str(last_valid_date))
ohlcv_array_dict = get_ohlcv_array_dict(complete_ohlcv_pdf)
# ASSERT NO ROW AND NO BAR IS DATED AFTER THE LAST VALIDATION DATE
assert check_last_date_bool([train_pdf["decision_ts"], valid_pdf["decision_ts"], complete_ohlcv_pdf["timestamp"]], last_valid_date)
# DEFINE THE FEATURE LIST (ABSOLUTE FEATURES EXCLUDED, AS STEPS 02-03)
feature_col_str_list = get_model_feature_col_str_list(train_pdf, exp_config.MODEL_FEATURE_SCALE_TYPE_LIST)
# DISPLAY INFORMATION
print(f"Train Rows:\t{len(train_pdf):,}\tPooled Validation Rows:\t{len(valid_pdf):,}\t(step 02: 60,144 and 5,964)")
print(f"Rows dated:\t{train_pdf['decision_ts'].min()} -> {valid_pdf['decision_ts'].max()}\tBars:\t{complete_ohlcv_pdf['timestamp'].min()} -> {complete_ohlcv_pdf['timestamp'].max()}")
print(f"Features ({len(feature_col_str_list)}):\t{feature_col_str_list}")
print(f"Distances ({len(exp_config.SIGNAL_CHECK_DELTA_LIST)}):\t{exp_config.SIGNAL_CHECK_DELTA_LIST}")
print(f"Model:\t{get_check_lgbm_param_dict()}\tweights: {exp_config.SAMPLE_WEIGHT_MODE}")

In [ ]:
# RECORD THE START TIME
start_time = time.time()
# CALL FUNCTION TO FIT THE REAL MODELS ON TRAINING AND MEASURE THEM ON THE POOLED VALIDATION (WITH INTERVALS)
real_dict = run_model_check_dict(train_pdf, valid_pdf, feature_col_str_list, ohlcv_array_dict, exp_config.SIGNAL_CHECK_DELTA_LIST)
metric_pdf = real_dict["metric_pdf"]
# DISPLAY INFORMATION
print(f"Real check processed in {(time.time() - start_time) / 60:.1f} minutes\tModels:\t{len(real_dict['model_dict'])}")
print(f"Real mean AUC over {len(metric_pdf)} distances:\t{real_dict['mean_auc']:.4f}")
# PREVIEW THE PER-DISTANCE MEASURES
metric_pdf.round(4)

In [ ]:
# RECORD THE START TIME
start_time = time.time()
# CALL FUNCTION TO RUN THE NULL CHECKS (FEATURES SHIFTED BY WHOLE SESSIONS, MODELS RETRAINED; ALL CORES, 1 LIGHTGBM THREAD PER MODEL)
print(f"Running {exp_config.SIGNAL_CHECK_NULL_RUN_COUNT} null checks in parallel ...")
null_summary_pdf, null_metric_pdf = get_null_run_pdf_tuple(train_pdf, valid_pdf, feature_col_str_list, ohlcv_array_dict, exp_config.SIGNAL_CHECK_DELTA_LIST)
# CALL FUNCTION TO GET THE VERDICT
verdict_dict = get_multivariate_verdict_dict(metric_pdf, null_summary_pdf)
# DISPLAY INFORMATION
print(f"Null checks processed in {(time.time() - start_time) / 60:.1f} minutes\n")
for key_str, value in verdict_dict.items():
    print(f"{key_str}:\t{value}")
print(f"\nVERDICT:\tOutcome {verdict_dict['outcome']} ({verdict_dict['outcome_str']})\tp = {verdict_dict['null_p_value']:.2f}")
print("Reminder: exp01 stays STOPPED whatever this outcome (diagnostic added after its results).")
# PREVIEW THE NULL RUNS
null_summary_pdf.round(4)

In [ ]:
# COMPARE EACH DISTANCE WITH ITS NULL DISTRIBUTION (DESCRIPTIVE, NOT PART OF THE VERDICT)
null_stat_pdf = null_metric_pdf.groupby("delta").agg(null_auc_mean=("auc", "mean"), null_auc_max=("auc", "max"), null_top_lift_mean=("top_lift", "mean"),
                                                     null_top_lift_max=("top_lift", "max")).reset_index()
distance_pdf = metric_pdf.merge(null_stat_pdf, on="delta", how="left")
distance_pdf["null_auc_ge_real_count"] = [int((null_metric_pdf.loc[null_metric_pdf["delta"] == row["delta"], "auc"] >= row["auc"]).sum()) for _, row in distance_pdf.iterrows()]
distance_pdf["null_top_lift_ge_real_count"] = [int((null_metric_pdf.loc[null_metric_pdf["delta"] == row["delta"], "top_lift"] >= row["top_lift"]).sum())
                                               for _, row in distance_pdf.iterrows()]
# DISPLAY THE PER-DISTANCE TABLE (REAL VS NULL; TOP DECILE VS BREAK-EVEN)
display(distance_pdf[["delta", "row_count", "base_rate", "auc", "auc_ci_low", "auc_ci_high", "null_auc_mean", "null_auc_max", "null_auc_ge_real_count"]].round(4))
distance_pdf[["delta", "top_row_count", "top_tp_rate", "top_tp_rate_ci_low", "top_tp_rate_ci_high", "top_breakeven_tp_rate", "clears_breakeven",
              "top_lift", "top_lift_ci_low", "top_lift_ci_high", "null_top_lift_mean", "null_top_lift_max", "null_top_lift_ge_real_count"]].round(4)

In [ ]:
# CALL FUNCTION TO GET THE CALIBRATION TABLE (DECILES OF PREDICTED P(TP) PER DISTANCE, POOLED; DESCRIPTIVE)
calibration_pdf = get_calibration_pdf(valid_pdf, real_dict["proba_pdf"], exp_config.SIGNAL_CHECK_DELTA_LIST)
display(calibration_pdf.round(4))
# CALL FUNCTION TO GET THE SPLIT-GAIN IMPORTANCE OF THE REAL MODELS (DESCRIPTIVE: WHAT THE MODELS USED, NOT EVIDENCE OF INFORMATION)
importance_pdf = get_feature_importance_pdf(real_dict["model_dict"], feature_col_str_list)
print(f"Top {exp_config.MULTIVARIATE_CHECK_IMPORTANCE_TOP_COUNT} features by mean split-gain share over the {len(real_dict['model_dict'])} models (descriptive):")
importance_pdf.head(exp_config.MULTIVARIATE_CHECK_IMPORTANCE_TOP_COUNT).round(4)

In [ ]:
# SAVE THE RESULTS
write_csv_file_to_path(distance_pdf, f"{output_path_str}distance_metric_data.csv", "W")
write_csv_file_to_path(null_summary_pdf, f"{output_path_str}null_run_data.csv", "W")
write_csv_file_to_path(null_metric_pdf, f"{output_path_str}null_distance_metric_data.csv", "W")
write_csv_file_to_path(calibration_pdf, f"{output_path_str}calibration_data.csv", "W")
write_csv_file_to_path(importance_pdf, f"{output_path_str}feature_importance_data.csv", "W")

In [ ]:
# DEFINE THE LOGGED WINDOWS (THE CHECK'S TRAINING AND POOLED VALIDATION; THE LATEST TEST WINDOW WAS NOT READ)
last_fold_row = pooled_fold_pdf.iloc[-1]
log_window_dict = {"fold_id": int(last_fold_row["fold_id"]), "train_start": window_dict["train_start"], "train_end": window_dict["train_end"],
                   "valid_start": window_dict["valid_start"], "valid_end": window_dict["valid_end"], "test_start": last_fold_row["test_start"], "test_end": last_fold_row["test_end"]}
# DEFINE THE PER-DISTANCE SUMMARIES
distance_summary_list = distance_pdf[["delta", "row_count", "base_rate", "auc", "auc_ci_low", "auc_ci_high", "null_auc_max", "top_tp_rate", "top_tp_rate_ci_low",
                                      "top_breakeven_tp_rate", "clears_breakeven", "top_lift", "top_lift_ci_low", "top_lift_ci_high"]].round(6).to_dict("records")
# LOG THE TRIAL (ONE ENTRY; THE NULL RUNS ARE PART OF IT)
multivariate_trial_dict = log_trial_dict(exp_config, exp_config.MULTIVARIATE_CHECK_STAGE_STR,
                                         {"train_years": window_dict["train_years"], "train_start": window_dict["train_start"], "train_end": window_dict["train_end"],
                                          "valid_fold_id_list": window_dict["valid_fold_id_list"], "valid_start": window_dict["valid_start"], "valid_end": window_dict["valid_end"],
                                          "sampling": [exp_config.SAMPLING_MODE, exp_config.SAMPLE_EVERY_N_MINUTES], "feature_list": feature_col_str_list,
                                          "delta_list": exp_config.SIGNAL_CHECK_DELTA_LIST, "lgbm_param_dict": get_check_lgbm_param_dict(),
                                          "sample_weight_mode": exp_config.SAMPLE_WEIGHT_MODE, "null_run_count": exp_config.SIGNAL_CHECK_NULL_RUN_COUNT,
                                          "null_shift_share_range": exp_config.SIGNAL_CHECK_NULL_SHIFT_SHARE_RANGE, "null_seed": config.RANDOM_SEED,
                                          "top_share": exp_config.MULTIVARIATE_CHECK_TOP_SHARE, "bootstrap": [exp_config.SIGNAL_CHECK_BOOTSTRAP_BLOCK_STR, config.BOOTSTRAP_ITERATION_COUNT]},
                                         {**verdict_dict, "distance_summary_list": distance_summary_list}, False, log_window_dict,
                                         note_str_in="exp01 step04 multivariate signal check (diagnostic added 2026-10-06 after exp01's STOP; cannot reverse it)")
print(f"Logged 1 '{exp_config.MULTIVARIATE_CHECK_STAGE_STR}' entry at {multivariate_trial_dict['logged_ts']} (config hash {get_config_hash_str(exp_config)})")

In [ ]:
# CALL FUNCTION TO READ THE TRIAL LOG
trial_log_pdf = read_trial_log_pdf()
print(f"Trial log rows:\t{len(trial_log_pdf):,}")
# DISPLAY THE NUMBER OF TRIALS PER EXPERIMENT AND STAGE (MULTIPLE-TESTING RECORD)
get_trial_count_pdf(trial_log_pdf)